In [58]:
import polars as pl
import numpy as np
from typing import Dict, List
import pickle
from pathlib import Path

import sys
sys.path.append("/kaggle/input/models/fedorfesenko/metrcs/other/default/1")
from metrics import evaluate_candidates

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F


In [2]:
df_train = pl.read_parquet("/kaggle/input/datasets/fedorfesenko/train-interactions/df_interactions_train.parquet")
df_val = pl.read_parquet("/kaggle/input/datasets/fedorfesenko/train-interactions/df_interactions_val.parquet")

In [3]:
df_train

user_id,product_id,action_type,timestamp,product_category
u64,u64,str,i64,str
7421231689159170748,15622873057108008619,"""AT_View""",1692527447,"""Колбаса сырокопченая/сыровялен…"
135364936461809128,17225686928570569253,"""AT_View""",1686479624,"""ПАСТА С МЯСОМ ПТИЦЫ"""
7846147652438475366,9571392871249924586,"""AT_View""",1691202307,"""Бумажные полотенца"""
365744836962066469,2281275230546064866,"""AT_Purchase""",1683883091,"""КОТЛЕТА МЯСНАЯ С ОВОЩАМИ"""
13306396713639957054,6458244614391609220,"""AT_View""",1691893320,"""Сок яблочный восстановленный о…"
…,…,…,…,…
12011850567211003172,11951705936862192916,"""AT_View""",1684231541,"""Творожки"""
8190566140622338193,8082895775749092273,"""AT_View""",1676117867,"""СЫРНИКИ КЛАССИЧЕСКИЕ"""
16811422400017691818,5695100139936509838,"""AT_View""",1685754860,"""Другие запеченые хлебцы"""


## Подготовка данных

Создаем истории пользователей в следующем формате:

```
{
    user_1: [item_A, item_B, item_C],
    user_2: [item_D, item_E],
}
```

In [4]:
(df_train
    .filter(pl.col("action_type") == "AT_CartUpdate")
    .group_by("user_id")
    .agg(pl.col("product_id").sort_by("timestamp"))
)

user_id,product_id
u64,list[u64]
13843528698353069763,"[678641909039548613, 9716635682142507904, 4141384210299998267]"
5309119020244314782,"[13210009803310626228, 18042177626338885367, … 15647675356834902484]"
15964009690442073426,"[3204947024121082676, 12125197384727620855, … 16703580308601396627]"
22421705853606406,[3704433686310748014]
169346398575821099,"[17155947748951669810, 9750367899800070943, … 94865538207334906]"
…,…
5806586791729190301,[15112030531332720997]
804642164988438294,"[2870343060070428851, 6248957849722053876, … 14852443199722945208]"
16643318569810029108,"[8860193103815737625, 7980048162444688783, … 12759425454439031747]"


In [5]:
train_histories = {
    row["user_id"]: row["items"]
    for row in 
    (df_train
    .filter(pl.col("action_type") == "AT_CartUpdate")
    .group_by("user_id")
    .agg(pl.col("product_id").sort_by("timestamp").alias("items"))).iter_rows(named=True)
    
}

In [6]:
len(train_histories)

2211

In [7]:
# делаем для validate

true_items_val = {
    row["user_id"]: set(row["items"])
    for row in (
        df_val
        .filter(pl.col("action_type") == "AT_CartUpdate")
        .group_by("user_id")
        .agg(pl.col("product_id").alias("items"))
    ).iter_rows(named=True)
}

val_targets = {
    uid: list(items)
    for uid, items in true_items_val.items()
}

In [8]:
len(val_targets)

432

In [9]:
all_users = set(train_histories.keys()) | set(val_targets.keys())
len(all_users)

2245

In [10]:
all_items = set()
for history in train_histories.values():
    all_items.update(history)
for targets in val_targets.values():
    all_items.update(targets)
len(all_items)

13179

In [11]:
user_to_idx = {uid: i for i, uid in enumerate(all_users)}
item_to_idx = {iid: i for i, iid in enumerate(all_items)}
idx_to_item = {i: item for item, i in item_to_idx.items()}

n_users = len(user_to_idx)
n_items = len(item_to_idx)

n_users, n_items

(2245, 13179)

In [12]:
train_histories_idx = {
    user_to_idx[uid]: [item_to_idx[iid] for iid in history]
    for uid, history in train_histories.items()
}

val_targets_idx = {
    user_to_idx[uid]: [item_to_idx[i] for i in items]
    for uid, items in val_targets.items()
    # if uid in set(train_histories.keys())
}

In [13]:
len(train_histories_idx), len(val_targets_idx)

(2211, 432)

In [14]:
# Популярность по cart-событиям
pop = (
    df_train
    .filter(pl.col("action_type") == "AT_CartUpdate")
    .group_by("product_id")
    .agg(pl.len().alias("n_carts"))
    .sort("n_carts", descending=True)
)

# Топ-500 product_id → конвертим в индексы
pop_items_id = pop.head(500)["product_id"].to_list()
pop_items_idx = [item_to_idx[i] for i in pop_items_id if i in item_to_idx]

print(f"pop_items_idx: {len(pop_items_idx)}")

pop_items_idx: 500


#### Режим 1: Train mode (`is_train=True`)

В режиме обучения датасет должен подготовить семплы на пользователя в постановке next-item prediction.

Если история пользователя: `[i1, i2, ..., iT]`, то нужно создать `T - 1` семплов. Для каждого `t` от `1` до `T-1` (позиция следующего айтема):

- `history` = префикс `history[:t]`, обрезанный до последних `max_seq_len` элементов
- `label` = следующий айтем `history[t]`

Формат train-семпла:
```python
{
  "uid": uid,
  "history": {
    "item_id": List[int],
    "length": int
  },
  "label": int
}
```

#### Режим 2: Inference mode (`is_train=False`)

В режиме оценки датасет должен возвращать ровно один семпл на пользователя.
Пользователь попадает в датасет только если для него есть таргеты в `labels`.
Содержимое семпла:
- `history` = история пользователя, обрезанная до последних `max_seq_len` элементов.

Формат eval-семпла:
```python
{
  "uid": uid,
  "history": {
    "item_id": List[int],
    "length": int
  }
}
```

In [15]:
class LavkaDataset(Dataset):
    def __init__(
        self,
        histories: Dict[int, List[int]],
        targets: Dict[int, List[int]],
        is_train: bool,
        max_seq_len: int = 100
    ):
        super().__init__()
        self.histories = histories
        self.targets = targets
        self.is_train = is_train
        self.max_seq_len = max_seq_len

        if self.is_train:
            self.samples = []
            for uid, history in histories.items():
                for t in range(1, len(history)):
                    self.samples.append((uid, t))
        else:
            self.users_with_targets = [
                uid for uid in targets.keys()
                if uid in histories
            ]
    def __getitem__(self, idx):
        if self.is_train:
            uid, t = self.samples[idx]
            history = self.histories[uid]
            prefix = history[:t][-self.max_seq_len:]
            label = history[t]
            return {
                "uid": uid,
                "history": {
                    "item_id": prefix,
                    "length": len(prefix)
                },
                "label": label
            }

        uid = self.users_with_targets[idx]
        history = self.histories[uid][-self.max_seq_len:]
        return {
                "uid": uid,
                "history": {
                    "item_id": history,
                    "length": len(history),
                }
            }
    def __len__(self):
        if self.is_train:
            return len(self.samples)
        return len(self.users_with_targets)
            

In [16]:
type(val_targets_idx)

dict

In [17]:
train_dataset = LavkaDataset(
    histories=train_histories_idx,
    targets=val_targets_idx,
    is_train=True
)

In [18]:
train_dataset[4]

{'uid': 210,
 'history': {'item_id': [7382, 6436, 5561, 6515, 6515], 'length': 5},
 'label': 7768}

In [19]:
eval_dataset = LavkaDataset(
    histories=train_histories_idx,
    targets=val_targets_idx,
    is_train=False
)

In [20]:
eval_dataset[78]

{'uid': 348,
 'history': {'item_id': [11468,
   2707,
   9360,
   6106,
   3966,
   2785,
   6268,
   6577,
   12605,
   12605,
   1396,
   1396,
   12173,
   12628,
   9260,
   12325,
   11070,
   11070,
   9630,
   6268,
   6268,
   10432,
   4109,
   9255,
   3170,
   627,
   12628,
   1463,
   1463,
   7397,
   2251,
   8628,
   2271,
   2447,
   10432,
   11461,
   6229,
   3592,
   538,
   2991,
   6609,
   6206,
   12325,
   10387,
   3066,
   3066,
   6440,
   3800,
   12325,
   9360,
   3066,
   11933,
   4846,
   4846,
   6609,
   4752,
   11454,
   10998,
   5243,
   6456,
   8364,
   11750,
   11750,
   11750,
   4462,
   8597,
   10998,
   10998,
   10998,
   10998,
   6843,
   1058,
   10743,
   9836,
   10998,
   11409,
   11056,
   5298,
   2013,
   11056,
   11056,
   6027,
   8597,
   4597,
   8597,
   9836,
   8922,
   1880,
   1024,
   4087,
   3239,
   4836,
   11750,
   11750,
   11750,
   11750,
   11863,
   7326,
   11863,
   4925],
  'length': 100}}

In [21]:
print(f"Train len: {len(train_dataset):,}")
print(f"Eval len: {len(eval_dataset):,}")

Train len: 299,331
Eval len: 398


In [22]:
def collate_fn(batch):
    uids = []
    items = []
    lengths = []
    labels = []

    is_train = "label" in batch[0]
    
    for sample in batch:
        uids.append(sample["uid"])
        items.extend(sample["history"]["item_id"])
        lengths.append(sample["history"]["length"])

        if is_train:
            labels.append(sample["label"])

    result = {
        "uid": torch.tensor(uids, dtype=torch.long),
        "history": {
            "item_id": torch.tensor(items, dtype=torch.long),
            "length": torch.tensor(lengths, dtype=torch.long)
        }
    }

    if is_train:
        result["label"] = torch.tensor(labels, dtype=torch.long)

    return result

In [23]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=4096,
    shuffle=True,
    collate_fn=collate_fn
)

In [24]:
eval_loader = DataLoader(
    dataset=eval_dataset,
    batch_size=4096,
    shuffle=False,
    collate_fn=collate_fn
)

## Base Two-Tower Model

Для начала сделаем башни, состоящие только из обучаемых эмбеддингов (по-сути это матричная факторизация с SGD)

In [25]:
class BaseTwoTowerModel(nn.Module):
    """
    Базовый класс двухбашенной модели, у которой башни - это обучаемые эмбеддинги
    """
    def __init__(self, n_users: int, n_items: int, emb_dim: int):
        super().__init__()
        self.user_embeds = nn.Embedding(n_users, emb_dim)
        self.item_embeds = nn.Embedding(n_items, emb_dim)

    def compute_loss(self, inputs):
        raise NotImplementedError

    def forward(self, inputs):
        if self.training:
            loss = self.compute_loss(inputs)
            return loss
        uid = inputs["uid"]
        user_emb = self.user_embeds(uid)
        item_embs = self.item_embeds.weight
        scores = user_emb @ item_embs.T
        return scores
        

## Base Two Tower Model with Softmax (full) loss

$$
\mathcal{L_{\text{softmax}}}=
-\sum_{u \in U}
\log{p(i^+|u)} = -\sum_{u \in U}\log{\frac{e^{r(u, i^+)}}{\sum_{d \in D}e^{r(u, d)}}}=
-\sum_{u \in U}\bigg[
r(u, i^+) - \log{\sum_{d \in D}e^{r(u, d)}}
\bigg] 
$$

In [26]:
class SoftmaxBaseTwoTowerModel(BaseTwoTowerModel):
    def compute_loss(self, inputs):
        user_emb = self.user_embeds(inputs["uid"])
        item_embs = self.item_embeds.weight
        scores = user_emb @ item_embs.T

        labels = inputs["label"]
        batch_size = scores.shape[0]
        positive_scores = scores[torch.arange(batch_size, device=scores.device), labels]
        logsumexp = torch.logsumexp(scores, dim=1)
        loss = -(positive_scores - logsumexp).mean()
        return loss

In [27]:
NUM_EPOCH = 20
EMB_DIM = 128
DEVICE = "cuda"

In [28]:
def to_device(batch, device):
    if isinstance(batch, dict):
        return {k: to_device(v, device) for k, v in batch.items()}
    elif isinstance(batch, torch.Tensor):
        return batch.to(device)
    return batch

In [29]:
def evaluate(
    model,
    dataloader,
    targets: Dict[int, List[int]], # {user_idx: [item_idx, ...]}
    catalog_size: int,
    pop_items_idx: List[int],
    topk: int = 500,
    device: str = "cuda"
) -> Dict[str, float]:

    model.eval()
    recommendations = {}

    with torch.no_grad():
        for batch in dataloader:
            batch = to_device(batch, device)
            
            scores = model(batch)
            _, topk_idx = scores.topk(topk, dim=1)

            for i, uid in enumerate(batch["uid"]):
                recommendations[uid.item()] = topk_idx[i].tolist()
    cold_users = [uid for uid in targets if uid not in recommendations]
    for uid in cold_users:
        recommendations[uid] = pop_items_idx

    recommendations_id = {
        uid: [idx_to_item[i] for i in items]
        for uid, items in recommendations.items()
    }
    targets_id = {
        uid: [idx_to_item[i] for i in items]
        for uid, items in targets.items()
    }
    
    # Вызываем готовый агрегатор
    result = evaluate_candidates(
        predicted_candidates=recommendations_id,
        true_items=targets_id,
        k=topk,
        total_items=catalog_size,
    )
    
    return result

In [30]:
def train(
    model,
    optimizer,
    train_dataloader,
    eval_dataloader,
    num_epochs,
    targets,
    catalog_size,
    pop_items_idx,
    topk=500,
    device="cuda"
):
    model.to(device)
    history = {"loss": [], "metrics": []}

    for epoch in range(num_epochs):
        model.train()
        cur_epoch_loss = 0
        
        for batch in train_dataloader:
            
            batch = to_device(batch, device)
            

            loss = model(batch) # так как  режим train, то forward вернет loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            cur_epoch_loss += loss.item()
            
        avg_loss = cur_epoch_loss / len(train_dataloader)
        history["loss"].append(avg_loss)
        metrics = evaluate(model, eval_dataloader, targets, catalog_size, pop_items_idx)
        history["metrics"].append(metrics)
        
        print(f"\nEpoch {epoch}: loss={avg_loss:.4f}")
        for name, value in metrics.items():
            if isinstance(value, float):
                print(f"  {name}: {value:.4f}")
            else:
                print(f"  {name}: {value}")
                
    return history

In [31]:
def init_weights(m):
    if isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, std=0.02)
    elif isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, std=0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [32]:
model_v1 = SoftmaxBaseTwoTowerModel(n_users, n_items, emb_dim=EMB_DIM)
model_v1.apply(init_weights)
optimizer_v1 = torch.optim.Adam(model_v1.parameters())

In [33]:
history_v1 = train(
    model=model_v1,
    optimizer=optimizer_v1,
    train_dataloader=train_loader,
    eval_dataloader=eval_loader,
    num_epochs=NUM_EPOCH,
    targets=val_targets_idx,
    catalog_size=n_items,
    pop_items_idx=pop_items_idx,
    topk=500,
    device=DEVICE,
)


Epoch 0: loss=9.3949
  recall@500: 0.3055
  hit_rate@500: 0.7778
  mrr@500: 0.1112
  coverage: 0.3221
  n_requests: 432
  avg_candidates: 500.0000

Epoch 1: loss=8.6967
  recall@500: 0.3103
  hit_rate@500: 0.7847
  mrr@500: 0.1018
  coverage: 0.2018
  n_requests: 432
  avg_candidates: 500.0000

Epoch 2: loss=8.3143
  recall@500: 0.3202
  hit_rate@500: 0.7963
  mrr@500: 0.1134
  coverage: 0.2213
  n_requests: 432
  avg_candidates: 500.0000

Epoch 3: loss=8.0986
  recall@500: 0.3388
  hit_rate@500: 0.7986
  mrr@500: 0.1524
  coverage: 0.2816
  n_requests: 432
  avg_candidates: 500.0000

Epoch 4: loss=7.8446
  recall@500: 0.3628
  hit_rate@500: 0.8032
  mrr@500: 0.2014
  coverage: 0.3513
  n_requests: 432
  avg_candidates: 500.0000

Epoch 5: loss=7.5780
  recall@500: 0.4122
  hit_rate@500: 0.8194
  mrr@500: 0.2411
  coverage: 0.4264
  n_requests: 432
  avg_candidates: 500.0000

Epoch 6: loss=7.3256
  recall@500: 0.4423
  hit_rate@500: 0.8403
  mrr@500: 0.2700
  coverage: 0.4974
  n_reque

In [34]:
metrics_v1 = evaluate(
    model=model_v1,
    dataloader=eval_loader,
    targets=val_targets_idx,
    catalog_size=n_items,
    pop_items_idx=pop_items_idx,
    topk=500,
    device="cuda",
)

In [35]:
metrics_v1

{'recall@500': 0.5681180143695442,
 'hit_rate@500': 0.8912037037037037,
 'mrr@500': 0.35045165207173984,
 'coverage': 0.8799605432885651,
 'n_requests': 432,
 'avg_candidates': 500.0}

## Base Two Tower Model with Softmax loss (Uniform Negatives)

In [36]:
class UniformSoftmaxBaseTwoTowerModel(BaseTwoTowerModel):
    def __init__(self, n_users, n_items, emb_dim, num_negatives=100):
        super().__init__(n_users, n_items, emb_dim)
        self.num_negatives = num_negatives
        
    def compute_loss(self, inputs):
        uid = inputs["uid"]
        batch_size = len(inputs["uid"])
        
        user_emb = self.user_embeds(uid)
        labels = inputs["label"]
        pos_embs = self.item_embeds(labels)
        
        # Uniform negatives
        neg_items = torch.randint(
            0, self.item_embeds.num_embeddings,
            (batch_size, self.num_negatives),
            device=uid.device
        )

        neg_embs = self.item_embeds(neg_items)

        pos_scores = (user_emb * pos_embs).sum(dim=-1, keepdim=True)   # (B, 1)
        neg_scores = (user_emb.unsqueeze(1) * neg_embs).sum(dim=-1)    # (B, K)
        
        # Логиты: [pos, neg1, neg2, ..., negK]
        logits = torch.cat([pos_scores, neg_scores], dim=1)   # (B, K+1)
        
        # Метки: позитив — всегда индекс 0
        targets = torch.zeros(batch_size, dtype=torch.long, device=uid.device)
        
        loss = F.cross_entropy(logits, targets)
        return loss

In [37]:
model_v2 = UniformSoftmaxBaseTwoTowerModel(
    n_users=n_users, n_items=n_items, emb_dim=EMB_DIM, num_negatives=1024
)
model_v2.apply(init_weights)
optimizer_v2 = torch.optim.Adam(model_v2.parameters())

In [38]:
history_v2 = train(
    model=model_v2,
    optimizer=optimizer_v2,
    train_dataloader=train_loader,
    eval_dataloader=eval_loader,
    num_epochs=NUM_EPOCH,
    targets=val_targets_idx,
    catalog_size=n_items,
    pop_items_idx=pop_items_idx,
    topk=500,
    device="cuda",
)


Epoch 0: loss=6.8583
  recall@500: 0.3266
  hit_rate@500: 0.7940
  mrr@500: 0.1379
  coverage: 0.3500
  n_requests: 432
  avg_candidates: 500.0000

Epoch 1: loss=6.1778
  recall@500: 0.3090
  hit_rate@500: 0.7894
  mrr@500: 0.1171
  coverage: 0.2539
  n_requests: 432
  avg_candidates: 500.0000

Epoch 2: loss=5.7078
  recall@500: 0.3328
  hit_rate@500: 0.8079
  mrr@500: 0.1400
  coverage: 0.2775
  n_requests: 432
  avg_candidates: 500.0000

Epoch 3: loss=5.4571
  recall@500: 0.3528
  hit_rate@500: 0.8125
  mrr@500: 0.1763
  coverage: 0.3399
  n_requests: 432
  avg_candidates: 500.0000

Epoch 4: loss=5.1972
  recall@500: 0.3844
  hit_rate@500: 0.8171
  mrr@500: 0.2182
  coverage: 0.4031
  n_requests: 432
  avg_candidates: 500.0000

Epoch 5: loss=4.9474
  recall@500: 0.4159
  hit_rate@500: 0.8194
  mrr@500: 0.2432
  coverage: 0.4634
  n_requests: 432
  avg_candidates: 500.0000

Epoch 6: loss=4.7216
  recall@500: 0.4390
  hit_rate@500: 0.8310
  mrr@500: 0.2707
  coverage: 0.5283
  n_reque

In [39]:
metrics_v2 = evaluate(
    model=model_v2,
    dataloader=eval_loader,
    targets=val_targets_idx,
    catalog_size=n_items,
    pop_items_idx=pop_items_idx,
    topk=500,
    device="cuda",
)

In [40]:
metrics_v2

{'recall@500': 0.5753582942016182,
 'hit_rate@500': 0.9050925925925926,
 'mrr@500': 0.35215889849465604,
 'coverage': 0.8877760072843159,
 'n_requests': 432,
 'avg_candidates': 500.0}

## Base Two Tower Model with Softmax loss (In-batch negatives)

In [62]:
class InBatchSoftmaxBaseTwoTowerModel(BaseTwoTowerModel):
    def compute_loss(self, inputs):
        uid = inputs["uid"]
        labels = inputs["label"]
        B = uid.shape[0]
        
        user_emb = self.user_embeds(uid)          
        item_embs = self.item_embeds(labels)      
        
        scores = user_emb @ item_embs.T           
        
        targets = torch.arange(B, device=uid.device)
        
        loss = F.cross_entropy(scores, targets)
        return loss

In [64]:
model_v3 = InBatchSoftmaxBaseTwoTowerModel(
    n_users=n_users, n_items=n_items, emb_dim=EMB_DIM)
model_v3.apply(init_weights)
optimizer_v3 = torch.optim.Adam(model_v3.parameters())

In [65]:
history_v3 = train(
    model=model_v3,
    optimizer=optimizer_v3,
    train_dataloader=train_loader,
    eval_dataloader=eval_loader,
    num_epochs=NUM_EPOCH,
    targets=val_targets_idx,
    catalog_size=n_items,
    pop_items_idx=pop_items_idx,
    topk=500,
    device="cuda",
)


Epoch 0: loss=8.2622
  recall@500: 0.3489
  hit_rate@500: 0.8032
  mrr@500: 0.2793
  coverage: 0.9958
  n_requests: 432
  avg_candidates: 500.0000

Epoch 1: loss=8.0742
  recall@500: 0.4249
  hit_rate@500: 0.8380
  mrr@500: 0.2704
  coverage: 0.8598
  n_requests: 432
  avg_candidates: 500.0000

Epoch 2: loss=7.6611
  recall@500: 0.4405
  hit_rate@500: 0.8449
  mrr@500: 0.2706
  coverage: 0.8465
  n_requests: 432
  avg_candidates: 500.0000

Epoch 3: loss=7.2547
  recall@500: 0.4456
  hit_rate@500: 0.8519
  mrr@500: 0.2786
  coverage: 0.8610
  n_requests: 432
  avg_candidates: 500.0000

Epoch 4: loss=6.9427
  recall@500: 0.4566
  hit_rate@500: 0.8565
  mrr@500: 0.2681
  coverage: 0.8816
  n_requests: 432
  avg_candidates: 500.0000

Epoch 5: loss=6.6989
  recall@500: 0.4576
  hit_rate@500: 0.8542
  mrr@500: 0.2722
  coverage: 0.8982
  n_requests: 432
  avg_candidates: 500.0000

Epoch 6: loss=6.5075
  recall@500: 0.4626
  hit_rate@500: 0.8611
  mrr@500: 0.2651
  coverage: 0.9151
  n_reque

In [70]:
model_v3.eval()
model_v3 = model_v3.to("cuda")
batch = next(iter(eval_loader))
batch = to_device(batch, "cuda")

with torch.no_grad():
    scores = model_v3(batch)

print(f"scores shape: {scores.shape}")

scores shape: torch.Size([398, 13179])


In [66]:
metrics_v3 = evaluate(
    model=model_v3,
    dataloader=eval_loader,
    targets=val_targets_idx,
    catalog_size=n_items,
    pop_items_idx=pop_items_idx,
    topk=500,
    device="cuda",
)

In [67]:
metrics_v3

{'recall@500': 0.4919578432351073,
 'hit_rate@500': 0.8726851851851852,
 'mrr@500': 0.15199692890463193,
 'coverage': 0.9727596934517034,
 'n_requests': 432,
 'avg_candidates': 500.0}

## User Tower with history

Башня пользователя будет выглядеть как __среднее__ эмбедов айтемов из его истории

In [46]:
class MeanUserEncoder(nn.Module):
    def __init__(self, n_items: int, emb_dim: int):
        super().__init__()
        self.item_embeddings = nn.Embedding(n_items, emb_dim)

    def forward(self, history: Dict[str, torch.Tensor]):
        item_idx = history["item_id"]
        item_embs = self.item_embeddings(item_idx)

        length = history["length"]
        batch_size = length.shape[0]

        user_indices = torch.repeat_interleave(
            torch.arange(batch_size, device=item_idx.device),
            length
        )   # (N,)
        
        emb_dim = self.item_embeddings.embedding_dim
        user_emb = torch.zeros(batch_size, emb_dim, device=item_idx.device)
        user_emb.scatter_add_(
            0,
            user_indices.unsqueeze(1).expand(-1, emb_dim),
            item_embs
        )
        
        user_emb = user_emb / length.unsqueeze(1)
        
        return user_emb   
        

## Two Tower Model with Mean User Encoder

In [47]:
class BaseMeanTwoTowerModel(nn.Module):
    def __init__(self, n_items, emb_dim):
        super().__init__()
        self.encoder = MeanUserEncoder(n_items, emb_dim)

    def compute_loss(self, inputs):
        raise NotImplementedError
    
    def forward(self, inputs):
        if self.training:
            return self.compute_loss(inputs)
        user_emb = self.encoder(inputs["history"])      
        item_embs = self.encoder.item_embeddings.weight  
        scores = user_emb @ item_embs.T
        return scores

## Two Tower Model with Mean User Encoder (full softmax loss)

In [48]:
class FullSoftmaxMeanTwoTowerModel(BaseMeanTwoTowerModel):
    def compute_loss(self, inputs):
        user_emb = self.encoder(inputs["history"])          
        item_embs = self.encoder.item_embeddings.weight     
        scores = user_emb @ item_embs.T                     
        
        labels = inputs["label"]
        return F.cross_entropy(scores, labels)

In [49]:
model_v4 = FullSoftmaxMeanTwoTowerModel(n_items, EMB_DIM)
model_v4.apply(init_weights)
optimizer_v4 = torch.optim.Adam(model_v4.parameters())

In [50]:
history_model_v4 = train(
    model_v4,
    optimizer_v4,
    train_loader,
    eval_loader,
    NUM_EPOCH,
    val_targets_idx,
    n_items,
    pop_items_idx
)


Epoch 0: loss=9.3810
  recall@500: 0.3010
  hit_rate@500: 0.7801
  mrr@500: 0.1088
  coverage: 0.2550
  n_requests: 432
  avg_candidates: 500.0000

Epoch 1: loss=8.7315
  recall@500: 0.3095
  hit_rate@500: 0.7917
  mrr@500: 0.1210
  coverage: 0.2112
  n_requests: 432
  avg_candidates: 500.0000

Epoch 2: loss=8.4080
  recall@500: 0.3268
  hit_rate@500: 0.7986
  mrr@500: 0.1127
  coverage: 0.2525
  n_requests: 432
  avg_candidates: 500.0000

Epoch 3: loss=8.2795
  recall@500: 0.3332
  hit_rate@500: 0.8032
  mrr@500: 0.1174
  coverage: 0.2930
  n_requests: 432
  avg_candidates: 500.0000

Epoch 4: loss=8.1582
  recall@500: 0.3529
  hit_rate@500: 0.8125
  mrr@500: 0.1392
  coverage: 0.3173
  n_requests: 432
  avg_candidates: 500.0000

Epoch 5: loss=8.0232
  recall@500: 0.3710
  hit_rate@500: 0.8218
  mrr@500: 0.1576
  coverage: 0.3314
  n_requests: 432
  avg_candidates: 500.0000

Epoch 6: loss=7.8792
  recall@500: 0.3899
  hit_rate@500: 0.8287
  mrr@500: 0.1680
  coverage: 0.3502
  n_reque

In [51]:
metrics_model_v4 = evaluate(
    model_v4,
    eval_loader,
    val_targets_idx,
    n_items,
    pop_items_idx
)
metrics_model_v4

{'recall@500': 0.5144845861843967,
 'hit_rate@500': 0.8703703703703703,
 'mrr@500': 0.32019822200026987,
 'coverage': 0.6434479095530768,
 'n_requests': 432,
 'avg_candidates': 500.0}

## In-batch negatives with LogQ-correction

In [52]:
class InBatchLogQMeanTwoTowerModel(BaseMeanTwoTowerModel):
    def __init__(self, n_items, emb_dim, item_counts):
        super().__init__(n_items, emb_dim)
        
        # q(i) = #i / #all
        item_probs = item_counts / item_counts.sum()
        log_q = torch.log(item_probs + 1e-10)
        
        self.register_buffer("log_q", log_q)
    
    def compute_loss(self, inputs):
        user_emb = self.encoder(inputs["history"])      # (B, dim)
        labels = inputs["label"]                         # (B,)
        item_embs = self.encoder.item_embeddings(labels) # (B, dim)
        
        scores = user_emb @ item_embs.T                 # (B, B)
        
        # LogQ correction: вычитаем log q для каждого item в батче
        log_q_batch = self.log_q[labels]                # (B,) — log q для labels батча
        scores = scores - log_q_batch.unsqueeze(0)      # (B, B) - (1, B)
        
        B = user_emb.shape[0]
        targets = torch.arange(B, device=user_emb.device)
        
        return F.cross_entropy(scores, targets)

In [53]:
item_counts_df = (
    df_train
    .filter(pl.col("action_type") == "AT_CartUpdate")
    .group_by("product_id")
    .agg(pl.len().alias("count"))
)

item_counts = torch.zeros(n_items, dtype=torch.float32, device="cuda")

for row in item_counts_df.iter_rows(named=True):
    if row["product_id"] in item_to_idx:
        item_counts[item_to_idx[row["product_id"]]] = row["count"]

print(f"item_counts: {item_counts.shape}")
print(f"max: {item_counts.max().item()}")
print(f"min: {item_counts.min().item()}")
print(f"nonzero: {(item_counts > 0).sum().item()}")

item_counts: torch.Size([13179])
max: 1134.0
min: 0.0
nonzero: 13106


In [54]:
model_v5 = InBatchLogQMeanTwoTowerModel(
    n_items=n_items,
    emb_dim=EMB_DIM,
    item_counts=item_counts,
)
model_v5.apply(init_weights)
optimizer_v5 = torch.optim.Adam(model_v5.parameters())

In [55]:
history_model_v5 = train(
    model_v5,
    optimizer_v5,
    train_loader,
    eval_loader,
    NUM_EPOCH,
    val_targets_idx,
    n_items,
    pop_items_idx
)


Epoch 0: loss=9.2097
  recall@500: 0.3227
  hit_rate@500: 0.7847
  mrr@500: 0.1319
  coverage: 0.2403
  n_requests: 432
  avg_candidates: 500.0000

Epoch 1: loss=8.6558
  recall@500: 0.3125
  hit_rate@500: 0.7870
  mrr@500: 0.1214
  coverage: 0.1779
  n_requests: 432
  avg_candidates: 500.0000

Epoch 2: loss=8.1666
  recall@500: 0.3322
  hit_rate@500: 0.8009
  mrr@500: 0.1171
  coverage: 0.2013
  n_requests: 432
  avg_candidates: 500.0000

Epoch 3: loss=7.9303
  recall@500: 0.3600
  hit_rate@500: 0.8148
  mrr@500: 0.1459
  coverage: 0.2383
  n_requests: 432
  avg_candidates: 500.0000

Epoch 4: loss=7.6979
  recall@500: 0.3955
  hit_rate@500: 0.8264
  mrr@500: 0.1609
  coverage: 0.2816
  n_requests: 432
  avg_candidates: 500.0000

Epoch 5: loss=7.4696
  recall@500: 0.4193
  hit_rate@500: 0.8356
  mrr@500: 0.2032
  coverage: 0.3213
  n_requests: 432
  avg_candidates: 500.0000

Epoch 6: loss=7.2674
  recall@500: 0.4450
  hit_rate@500: 0.8426
  mrr@500: 0.2239
  coverage: 0.3587
  n_reque

In [56]:
metrics_model_v5 = evaluate(
    model_v5,
    eval_loader,
    val_targets_idx,
    n_items,
    pop_items_idx
)
metrics_model_v5

{'recall@500': 0.543822656362749,
 'hit_rate@500': 0.8842592592592593,
 'mrr@500': 0.3647542592625904,
 'coverage': 0.6200015175658244,
 'n_requests': 432,
 'avg_candidates': 500.0}

## Base Two Tower Model with Softmax loss (In-batch negatives) + logQ-correction 

In [76]:
class InBatchLogQMF(BaseTwoTowerModel):
    def __init__(self, n_users, n_items, emb_dim, item_counts):
        super().__init__(n_users, n_items, emb_dim)
        
        # LogQ
        item_probs = item_counts / item_counts.sum()
        log_q = torch.log(item_probs + 1e-10)
        self.register_buffer("log_q", log_q)
    
    def compute_loss(self, inputs):
        # User и позитивные item эмбеддинги
        user_emb = self.user_embeds(inputs["uid"])    # (B, dim)
        labels = inputs["label"]
        item_embs = self.item_embeds(labels)             # (B, dim)
        
        # Scores (B, B)
        scores = user_emb @ item_embs.T
        
        # LogQ correction
        log_q_batch = self.log_q[labels]              # (B,)
        scores = scores - log_q_batch.unsqueeze(0)    # (B, B) - (1, B)
        
        # Targets — диагональ
        B = user_emb.shape[0]
        targets = torch.arange(B, device=user_emb.device)
        
        return F.cross_entropy(scores, targets)

In [77]:
model_v6 = InBatchLogQMF(
    n_users=n_users,
    n_items=n_items,
    emb_dim=EMB_DIM,
    item_counts=item_counts,
)
model_v6.apply(init_weights)

optimizer_v6 = torch.optim.Adam(model_v6.parameters())

In [78]:
history_v6 = train(
    model=model_v6,
    optimizer=optimizer_v6,
    train_dataloader=train_loader,
    eval_dataloader=eval_loader,
    num_epochs=NUM_EPOCH,
    targets=val_targets_idx,
    catalog_size=n_items,
    pop_items_idx=pop_items_idx,
    topk=500,
    device="cuda",
)


Epoch 0: loss=9.2391
  recall@500: 0.3751
  hit_rate@500: 0.8032
  mrr@500: 0.2699
  coverage: 0.8689
  n_requests: 432
  avg_candidates: 500.0000

Epoch 1: loss=8.9539
  recall@500: 0.3679
  hit_rate@500: 0.8032
  mrr@500: 0.2109
  coverage: 0.4655
  n_requests: 432
  avg_candidates: 500.0000

Epoch 2: loss=8.1524
  recall@500: 0.3787
  hit_rate@500: 0.8171
  mrr@500: 0.2051
  coverage: 0.3968
  n_requests: 432
  avg_candidates: 500.0000

Epoch 3: loss=7.5283
  recall@500: 0.4071
  hit_rate@500: 0.8218
  mrr@500: 0.2477
  coverage: 0.4160
  n_requests: 432
  avg_candidates: 500.0000

Epoch 4: loss=7.1706
  recall@500: 0.4391
  hit_rate@500: 0.8403
  mrr@500: 0.2727
  coverage: 0.4626
  n_requests: 432
  avg_candidates: 500.0000

Epoch 5: loss=6.9055
  recall@500: 0.4639
  hit_rate@500: 0.8472
  mrr@500: 0.2871
  coverage: 0.5045
  n_requests: 432
  avg_candidates: 500.0000

Epoch 6: loss=6.6997
  recall@500: 0.4880
  hit_rate@500: 0.8565
  mrr@500: 0.2882
  coverage: 0.5422
  n_reque

In [79]:
metrics_model_v6 = evaluate(
    model_v6,
    eval_loader,
    val_targets_idx,
    n_items,
    pop_items_idx
)
metrics_model_v6

{'recall@500': 0.5635557472568301,
 'hit_rate@500': 0.8796296296296297,
 'mrr@500': 0.3508338440234694,
 'coverage': 0.8285150618408074,
 'n_requests': 432,
 'avg_candidates': 500.0}

## Итоговое сравнение моделей

In [81]:
models_info = {
    "v1": {"model": model_v1, "type": "MF full softmax"},
    "v2": {"model": model_v2, "type": "MF uniform K=1024"},
    "v3": {"model": model_v3, "type": "MF in-batch"},
    "v4": {"model": model_v4, "type": "Mean Two-Tower full"},
    "v5": {"model": model_v5, "type": "Mean Two-Tower in-batch + LogQ"},
    "v6": {"model": model_v6, "type": "MF in-batch + LogQ"},
}

all_metrics = []

for name, info in models_info.items():
    metrics = evaluate(
        info["model"],
        eval_loader,
        val_targets_idx,
        n_items,
        pop_items_idx,
        topk=500,
        device="cuda",
    )
    metrics["model"] = name
    metrics["type"] = info["type"]
    metrics["epochs"] = NUM_EPOCH
    metrics["dim"] = EMB_DIM
    all_metrics.append(metrics)

df_metrics = pl.DataFrame(all_metrics)

# Худшая по recall
worst_recall = df_metrics["recall@500"].min()

# Прирост
df_metrics = df_metrics.with_columns(
    (
        pl.lit("+") +
        ((pl.col("recall@500") - worst_recall) / worst_recall * 100)
        .round(2)
        .cast(pl.Utf8) +
        pl.lit("%")
    ).alias("Δ recall")
)

# Форматирование
df_formatted = (
    df_metrics
    .select([
        "model",
        "type",
        "epochs",
        "dim",
        pl.col("recall@500").round(4).alias("recall"),
        "Δ recall",
        pl.col("hit_rate@500").round(4).alias("hit_rate"),
        pl.col("mrr@500").round(4).alias("mrr"),
        pl.col("coverage").round(4).alias("coverage"),
        "n_requests",
    ])
    .sort("recall", descending=True)
)

df_formatted

model,type,epochs,dim,recall,Δ recall,hit_rate,mrr,coverage,n_requests
str,str,i64,i64,f64,str,f64,f64,f64,i64
"""v2""","""MF uniform K=1024""",20,128,0.5754,"""+16.95%""",0.9051,0.3522,0.8878,432
"""v1""","""MF full softmax""",20,128,0.5681,"""+15.48%""",0.8912,0.3505,0.88,432
"""v6""","""MF in-batch + LogQ""",20,128,0.5636,"""+14.55%""",0.8796,0.3508,0.8285,432
"""v5""","""Mean Two-Tower in-batch + LogQ""",20,128,0.5438,"""+10.54%""",0.8843,0.3648,0.62,432
"""v4""","""Mean Two-Tower full""",20,128,0.5145,"""+4.58%""",0.8704,0.3202,0.6434,432
"""v3""","""MF in-batch""",20,128,0.492,"""+0.0%""",0.8727,0.152,0.9728,432


## Two Tower эксперименты 

| model | type | recall@500 | mrr@500 | coverage |
|---|---|---|---|---|
| v2 | MF uniform K=1024 | __0.5754__ | __0.3522__ | 0.8878 |
| v1 | MF full softmax | 0.5681 | 0.3505 | 0.8800 |
| v6 | MF in-batch + LogQ | 0.5636 | 0.3508 | 0.8285 |
| v5 | Mean Two-Tower in-batch + LogQ | 0.5438 | 0.3648 | 0.6200 |
| v4 | Mean Two-Tower full | 0.5145 | 0.3202 | 0.6434 |
| v3 | MF in-batch (no LogQ) | 0.4920 | 0.1520 | __0.9728__ |

In [83]:
## сохраняем v5, v6

In [86]:
Path("/kaggle/working/artifacts").mkdir(parents=True, exist_ok=True)

# v5: Mean Two-Tower in-batch + LogQ 
torch.save({
    "model_state_dict": model_v5.state_dict(),
    "model_class": "InBatchLogQMeanTwoTowerModel",
    "config": {
        "n_items": n_items,
        "emb_dim": EMB_DIM,
    },
    "user_to_idx": user_to_idx,
    "item_to_idx": item_to_idx,
    "idx_to_item": idx_to_item,
    "n_users": n_users,
    "n_items": n_items,
    "pop_items": pop_items_idx,
}, "/kaggle/working/artifacts/two_tower_v5_logq.pth")

print("v5 сохранён")

# v6: MF in-batch + LogQ 
torch.save({
    "model_state_dict": model_v6.state_dict(),
    "model_class": "InBatchLogQMF",
    "config": {
        "n_users": n_users,
        "n_items": n_items,
        "emb_dim": EMB_DIM,
    },
    "user_to_idx": user_to_idx,
    "item_to_idx": item_to_idx,
    "idx_to_item": idx_to_item,
    "n_users": n_users,
    "n_items": n_items,
    "pop_items": pop_items_idx,
}, "/kaggle/working/artifacts/mf_v6_inbatch_logq.pth")

print("v6 сохранён")

v5 сохранён
v6 сохранён
